# Zero-shot baseline (ConvNeXt-Base CLIP)

The simplest possible baseline: **no training and no example images**.

**Model:** ConvNeXt-Base CLIP (`convnext_base`, trained by LAION on LAION-400M). It uses a convolutional image encoder (no transformer), a 512-dimensional embedding and 224px images, which makes it fast enough to run many experiments on a laptop.

1. Turn each of the 16 class names into one text prompt: `"a photo of a {class}."`
2. Embed the prompts with the text encoder and every image with the image encoder.
3. Predict the class whose text vector is most similar (cosine similarity) to the image vector.

**Data:** zero-shot never trains, so every labeled image is unseen by the model. We evaluate on both labeled splits, `data/train` (2,400 images) and `data/test` (400 images). `data/test2` has no labels, so it can't be scored.

Every later method (image prototypes, prompt ensembles, fine-tuning) has to beat these numbers.

In [ ]:
import re
import time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

import open_clip  # the local copy in ./open_clip

# ---- settings ----
MODEL_NAME = "convnext_base"
PRETRAINED = "laion400m_s13b_b51k"
SPLITS = {"train": Path("data/train"), "test": Path("data/test")}   # labeled splits, one sub-folder per class
PROMPT = "a photo of a {}."
BATCH_SIZE = 32

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Load the model
The first run downloads the weights from Hugging Face (~0.7 GB) and caches them.

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model = model.to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"{MODEL_NAME}: {n_params / 1e6:.0f}M parameters")

## 2. Load the images
`ImageFolder` uses the folder names as labels, sorted alphabetically. Both splits have the same 16 class folders, so the label numbers match.

In [ ]:
datasets = {name: ImageFolder(path, transform=preprocess) for name, path in SPLITS.items()}

folder_names = datasets["test"].classes
for name, ds in datasets.items():
    assert ds.classes == folder_names, f"{name} has different class folders"
    print(f"{name:5s}: {len(ds)} images, {len(ds.classes)} classes")

## 3. Turn folder names into readable class names
`LivingRoom` becomes `living room`, `TallBuilding` becomes `tall building`, and so on. The text encoder understands real words much better than CamelCase.

In [ ]:
def readable(name):
    return re.sub(r"(?<=[a-z])(?=[A-Z])", " ", name).lower()

class_names = [readable(n) for n in folder_names]
prompts = [PROMPT.format(c) for c in class_names]
for f, p in zip(folder_names, prompts):
    print(f"{f:14s} -> {p}")

## 4. Embed the prompts (one vector per class)

In [ ]:
with torch.no_grad():
    text_tokens = tokenizer(prompts).to(device)
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)   # unit length

print("text features:", tuple(text_features.shape))   # (16 classes, embedding size)

## 5. Embed every image and pick the most similar class

In [ ]:
def zero_shot_predict(dataset):
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            image_features = model.encode_image(images.to(device))
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

            similarity = image_features @ text_features.T     # cosine similarity, shape (batch, 16)
            all_preds.append(similarity.argmax(dim=-1).cpu())
            all_labels.append(labels)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()

results = {}
for name, ds in datasets.items():
    start = time.time()
    results[name] = zero_shot_predict(ds)
    print(f"{name:5s}: embedded {len(ds)} images in {time.time() - start:.0f}s")

## 6. Results

In [ ]:
print(f"Zero-shot accuracy ({MODEL_NAME}, prompt '{PROMPT}')")
for name, (preds, labels) in results.items():
    print(f"  {name:5s}: {accuracy_score(labels, preds):.2%}  ({len(labels)} images)")

all_preds = np.concatenate([p for p, _ in results.values()])
all_labels = np.concatenate([l for _, l in results.values()])
print(f"  all  : {accuracy_score(all_labels, all_preds):.2%}  ({len(all_labels)} images)")

### Per-class accuracy (recall) on each split

In [ ]:
cm_by_split = {name: confusion_matrix(labels, preds, labels=range(len(folder_names))) for name, (preds, labels) in results.items()}
per_class = {name: cm.diagonal() / cm.sum(axis=1) for name, cm in cm_by_split.items()}

print(f"{'class':14s}" + "".join(f"{n:>8s}" for n in per_class))
for i, f in enumerate(folder_names):
    print(f"{f:14s}" + "".join(f"{per_class[n][i]:8.0%}" for n in per_class))

### Full report for the train split (largest, most reliable numbers)

In [ ]:
preds, labels = results["train"]
print(classification_report(labels, preds, target_names=folder_names, digits=3))

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(9 * len(results), 8))
for ax, (name, (preds, labels)) in zip(np.atleast_1d(axes), results.items()):
    acc = accuracy_score(labels, preds)
    ConfusionMatrixDisplay(cm_by_split[name], display_labels=folder_names).plot(ax=ax, xticks_rotation=90, colorbar=False, cmap="Blues")
    ax.set_title(f"{name}: {acc:.1%}")
fig.suptitle(f"Zero-shot confusion matrices: {MODEL_NAME}")
plt.tight_layout()
plt.show()